# Tool-Discovery Agent + Biomni Integration

在 Colab 中运行 MCP server 并对接 Biomni Agent。

**流程:** 安装依赖 → 启动 MCP Server → 运行 Discovery Pipeline → Biomni 调用发现的工具

In [ ]:
!pip install -q requests beautifulsoup4 pyyaml pandas tabulate python-dateutil biomni langchain-openai

## Step 1: Clone repo & setup

In [ ]:
import os, subprocess, time, signal, sys

REPO_DIR = "/content/scientific_training2_cxy"
if not os.path.exists(REPO_DIR):
    !cd /content && git clone https://github.com/caixiaoyao2025/scientific_training2_cxy.git

os.chdir(REPO_DIR)
os.makedirs("data", exist_ok=True)
print(f"Working in: {os.getcwd()}")

## Step 2: Start MCP Server (stdio, no Docker)

In [ ]:
env = os.environ.copy()
env["MCP_DATA_ROOT"] = os.path.join(os.getcwd(), "data")
env["MCP_APP_ROOT"] = os.getcwd()

server_proc = subprocess.Popen(
    [sys.executable, "server.py"],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    env=env, text=True
)
time.sleep(2)
print(f"MCP server started, PID={server_proc.pid}")
print(f"Return code: {server_proc.poll()}")

## Step 3: Run Discovery Pipeline

In [ ]:
from run_pipeline import run_full_pipeline
run_full_pipeline(query="bioinformatics protein structure prediction tools", max_results=5)

## Step 4: Connect Biomni to MCP Server

In [ ]:
# Stop the standalone server first (Biomni will manage its own)
server_proc.terminate()
server_proc.wait()
print("Standalone server stopped.")

In [ ]:
# Write MCP config for Colab (no Docker, use Python directly)
import yaml

config = {
    "mcp_servers": {
        "bio-mcp": {
            "command": [sys.executable, os.path.join(os.getcwd(), "server.py")],
            "env": {
                "MCP_DATA_ROOT": os.path.join(os.getcwd(), "data"),
                "MCP_APP_ROOT": os.getcwd(),
            }
        }
    }
}

config_path = os.path.join(os.getcwd(), "mcp_config_colab.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f)
print(f"Config written to {config_path}")

In [ ]:
from biomni.agent import A1

agent = A1(
    path=os.path.join(os.getcwd(), "data"),
    llm="Qwen/Qwen2.5-14B-Instruct",
    source="Custom",
    base_url="https://api.siliconflow.cn/v1",
    api_key="sk-lufftravmhzgdpudfvbvzwrlfyctebizytmtlbynyiohtkij",
    expected_data_lake_files=[],
)
agent.add_mcp(config_path=config_path)

result = agent.go("Use list_registered_tools tool to show all available tools")
print(result)

## (Optional) Use HTTP mode instead

In [ ]:
# Alternative: run MCP server in HTTP mode, connect via URL
env_http = os.environ.copy()
env_http["MCP_DATA_ROOT"] = os.path.join(os.getcwd(), "data")
env_http["MCP_APP_ROOT"] = os.getcwd()
env_http["MCP_TRANSPORT"] = "streamable-http"
env_http["MCP_HOST"] = "0.0.0.0"
env_http["MCP_PORT"] = "8765"
env_http["MCP_PATH"] = "/mcp"

server_http = subprocess.Popen(
    [sys.executable, "server.py"],
    env=env_http, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
time.sleep(3)
print(f"HTTP MCP server started on port 8765, PID={server_http.pid}")